In [1]:
from dotenv import load_dotenv
import os
import requests
from requests.auth import HTTPBasicAuth
import openmeteo_requests
import requests_cache
from retry_requests import retry
import pandas as pd

In [5]:
load_dotenv(override=True)

CITY = os.getenv("CITY")
USERNAME = os.getenv("USERNAME")
PASSWORD = os.getenv("PASSWORD")
BASE_URL = f"https://{CITY}.pulse.eco/rest"

print(f"City: {CITY}")


City: skopje


In [6]:
def get_sensors():
    url = f"{BASE_URL}/sensor"
    r = requests.get(url, auth=HTTPBasicAuth(USERNAME, PASSWORD))
    if r.status_code != 200:
        raise Exception("Failed to fetch sensors:", r.text)
    return r.json()

valid_statuses = {
    "ACTIVE",
    "ACTIVE_UNCONFIRMED",
    "NOT_CLAIMED",
    "NOT_CLAIMED_UNCONFIRMED"
}


sensors = get_sensors()
filtered = [s for s in sensors if s["status"] in valid_statuses]

print("Total sensors:", len(sensors))
print("Filtered sensors:", len(filtered))
print("Example sensor:", filtered[0])

Total sensors: 195
Filtered sensors: 186
Example sensor: {'sensorId': 'sensor_dev_60237_141', 'position': '42.03900255426,21.40771061182', 'comments': 'Imported Sensor.community #60237', 'type': '20004', 'description': 'Sensor.community 60237', 'status': 'NOT_CLAIMED'}


In [7]:
sensor_locations = []

for s in filtered:
    lat, lon = map(float, s["position"].split(","))
    
    sensor_locations.append({
        "sensorId": s["sensorId"],
        "lat": lat,
        "lon": lon
    })

print("Sensors with coordinates:", len(sensor_locations))
sensor_locations[:3]

Sensors with coordinates: 186


[{'sensorId': 'sensor_dev_60237_141',
  'lat': 42.03900255426,
  'lon': 21.40771061182},
 {'sensorId': 'sensor_dev_10699_244', 'lat': 41.986, 'lon': 21.452},
 {'sensorId': 'fefbf9e0-ff44-4b85-b968-2af046c4f4dc',
  'lat': 42.01381277078925,
  'lon': 21.383400684279913}]

In [8]:
cache_session = requests_cache.CachedSession(".cache", expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)

openmeteo = openmeteo_requests.Client(session=retry_session)

url = "https://archive-api.open-meteo.com/v1/archive"

all_weather = []

for sensor in sensor_locations:
    
    params = {
        "latitude": sensor["lat"],
        "longitude": sensor["lon"],
        "start_date": "2025-12-01",
        "end_date": "2026-03-01",
        "hourly": [
            "temperature_2m",
            "relative_humidity_2m",
            "wind_speed_10m",
            "wind_direction_10m",
            "surface_pressure"
        ],
        "timezone": "Europe/Skopje"
    }

    responses = openmeteo.weather_api(url, params=params)
    response = responses[0]

    hourly = response.Hourly()

    timestamps = pd.date_range(
        start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
        end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
        freq=pd.Timedelta(seconds=hourly.Interval()),
        inclusive="left"
    )

    df = pd.DataFrame({
        "timestamp": timestamps,
        "sensorId": sensor["sensorId"],
        "lat": sensor["lat"],
        "lon": sensor["lon"],
        "temperature_2m": hourly.Variables(0).ValuesAsNumpy(),
        "relative_humidity_2m": hourly.Variables(1).ValuesAsNumpy(),
        "wind_speed_10m": hourly.Variables(2).ValuesAsNumpy(),
        "wind_direction_10m": hourly.Variables(3).ValuesAsNumpy(),
        "surface_pressure": hourly.Variables(4).ValuesAsNumpy(),
    })

    all_weather.append(df)

    print("Fetched weather for", sensor["sensorId"])

Fetched weather for sensor_dev_60237_141
Fetched weather for sensor_dev_10699_244
Fetched weather for fefbf9e0-ff44-4b85-b968-2af046c4f4dc
Fetched weather for sensor_dev_81984_843
Fetched weather for sensor_dev_78082_739
Fetched weather for 1fc02db8-38d2-4bb5-a2cb-a72f357d0e51
Fetched weather for 1002
Fetched weather for d42147ab-9f8c-4816-964b-62e6a82d6492
Fetched weather for e1f23667-3ff3-475b-9931-3f4de249b5b8
Fetched weather for 52546b6e-77bf-40f8-b0e5-6e54d7947e9f
Fetched weather for b17885fb-df39-477c-ba24-627edccb3c70
Fetched weather for 11888f3a-bc5e-4a0c-9f27-702984decedf
Fetched weather for 01440b05-255d-4764-be87-bdf135f32289
Fetched weather for 3791738b-bec6-452e-9aae-5b11a899bfe2
Fetched weather for bb948861-3fd7-47dd-b986-cb13c9732725
Fetched weather for 1a2af884-336b-427d-9b37-fe332557539f
Fetched weather for f9e27604-2715-4ec3-ac6c-e3385e29590c
Fetched weather for fef6bc74-bf86-4874-9531-51b033580379
Fetched weather for sensor_dev_84941_208
Fetched weather for 6c6a9ef6-

In [9]:
weather_df = pd.concat(all_weather, ignore_index=True)

print(weather_df.shape)
weather_df.head()

(406224, 9)


,timestamp,sensorId,lat,lon,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,surface_pressure
0,2025-11-30 22:00:00+00:00,sensor_dev_60237_141,42.039003,21.407711,5.2365,86.926003,0.648999,123.690094,982.719727
1,2025-11-30 23:00:00+00:00,sensor_dev_60237_141,42.039003,21.407711,4.2865,89.018982,0.569210,108.435043,982.986633
2,2025-12-01 00:00:00+00:00,sensor_dev_60237_141,42.039003,21.407711,3.6865,89.286163,0.763675,315.000092,983.007507
3,2025-12-01 01:00:00+00:00,sensor_dev_60237_141,42.039003,21.407711,3.2365,89.889656,1.781909,315.000092,983.240234
4,2025-12-01 02:00:00+00:00,sensor_dev_60237_141,42.039003,21.407711,2.7865,89.532578,2.930188,312.510406,983.086731


In [10]:
weather_df.to_csv("../data/streaming/skopje_sensor_weather_features_online.csv", index=False)